In [2]:
# init
round3 <- function(x) round(x, 3)

url <- paste0(
  "https://raw.githubusercontent.com/",
  "cshsiehecon/SPE7002/main/data/MROZ.csv"
)

df <- read.csv(url)
print(names(df))

 [1] "inlf"     "hours"    "kidslt6"  "kidsge6"  "age"      "educ"    
 [7] "wage"     "repwage"  "hushrs"   "husage"   "huseduc"  "huswage" 
[13] "faminc"   "mtr"      "motheduc" "fatheduc" "unem"     "city"    
[19] "exper"    "nwifeinc" "lwage"    "expersq" 


# Part 1

## Number 2

In [3]:
# only consider those in the labor force and with wage
earn <- subset(df, inlf == 1 & !is.na(lwage))

## Short
short <- lm(lwage ~ educ, data = earn)
sum <- summary(short)
coef <- coef(sum)
se <- sqrt(diag(vcov(short)))

df_short <- data.frame (
  Variable = rownames(coef),
  Coefficient = round3(coef[, "Estimate"]),
  Std_Error = round3(se),
  P_Value = round3(coef[, "Pr(>|t|)"])
)
cat("short regression:\n")
print(df_short)

# ## Mincer
mincer <- lm(lwage ~ educ + exper + expersq, data = earn)

sum_minc <- summary(mincer)
coef_minc <- coef(sum_minc)
se_minc <- sqrt(diag(vcov(mincer)))

df_mincer <- data.frame (
  Variable = rownames(coef_minc),
  Coefficient = round3(coef_minc[, "Estimate"]),
  Std_Error = round3(se_minc),
  P_Value = round3(coef_minc[, "Pr(>|t|)"])
)

cat("Mincer regression:\n")
print(df_mincer)


short regression:
               Variable Coefficient Std_Error P_Value
(Intercept) (Intercept)      -0.185     0.185   0.318
educ               educ       0.109     0.014   0.000
Mincer regression:
               Variable Coefficient Std_Error P_Value
(Intercept) (Intercept)      -0.522     0.199   0.009
educ               educ       0.107     0.014   0.000
exper             exper       0.042     0.013   0.002
expersq         expersq      -0.001     0.000   0.040


Because exper and expersq are statistically significant, they appear to contain information about lwage in this sample. So including them may improve the model compared with the short regression.

## Number 3

In [9]:
educ_estimate <- coef_minc["educ","Estimate"]
usual_coef <- 100 * educ_estimate
exact_percentage_change <- 100 * (exp(educ_estimate) - 1)

print(paste("Usual coefficient: ", round3(usual_coef), "%, Exact percentage change: ", round3(exact_percentage_change), "%"))

[1] "Usual coefficient:  10.749 %, Exact percentage change:  11.348 %"


Holding exper and expersq constant, using usual coefficient, 1 year of additional education increase wage by 10.749% and using exact percentage change increase wage by 11.348%

## Number 4

### prove X'e = 0

In [13]:
X <- model.matrix(mincer)
e <- residuals(mincer)

res <- t(X) %*% e
print(round3(res))

            [,1]
(Intercept)    0
educ           0
exper          0
expersq        0


### Frisch–Waugh–Lovell

In [ ]:
lwage_removed <- lm(lwage ~ exper + expersq, data = earn)
lwage_resid <- residuals(lwage_removed)

educ_controls <- lm(educ ~ exper + expersq, data = earn)
educ_resid <- residuals(educ_controls)

res <- lm(lwage_resid ~ educ_resid)

print(res)

ERROR: Error: object 'lwage_controls' not found
